# Chapter 10.1: Architecture Comparison - vLLM vs SGLang Design Philosophy

This notebook provides a deep, hands-on comparison of the architectural designs of **vLLM** and **SGLang**, two leading LLM serving systems. We will trace requests through both systems, compare components side-by-side, and build visual tools to understand the differences.

## Learning Objectives
- Understand the design philosophy behind each system
- Map components between vLLM and SGLang
- Trace request flow through both architectures
- Identify key architectural trade-offs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part10/chapter_10.1_architecture.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part10/chapter_10.1_architecture.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
# Setup and imports
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
from enum import Enum
import time
import json

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
print("Environment ready for architecture comparison.")

## 1. Design Philosophy Overview

| Aspect | vLLM | SGLang |
|--------|------|--------|
| **Core Philosophy** | Systems-first: optimize the serving engine | Co-design: optimize frontend + backend together |
| **Origin** | UC Berkeley (2023) | Stanford / UC Berkeley (2024) |
| **Key Innovation** | PagedAttention | RadixAttention + Frontend DSL |
| **Primary Goal** | High-throughput batched serving | Efficient complex LLM programs |
| **Architecture Style** | Monolithic engine with pluggable components | Layered: Router -> Scheduler -> Runtime |

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np

def draw_side_by_side_architecture_highlighted():
    """Side-by-side vLLM vs SGLang architecture with key differences highlighted in red."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 14))

    def draw_stack(ax, title, title_color, components, diff_indices):
        ax.set_xlim(0, 10); ax.set_ylim(0, 16); ax.axis('off')
        ax.set_title(title, fontsize=16, fontweight='bold', color=title_color, pad=15)
        for i, (label, color) in enumerate(components):
            y = 14 - i * 1.6
            is_diff = i in diff_indices
            ec = '#E74C3C' if is_diff else '#333'
            lw = 3.5 if is_diff else 1.5
            rect = FancyBboxPatch((0.5, y), 9, 1.3, boxstyle='round,pad=0.1',
                                  facecolor=color, edgecolor=ec, linewidth=lw)
            ax.add_patch(rect)
            ax.text(5, y + 0.65, label, ha='center', va='center',
                    fontsize=10, fontweight='bold', color='white' if color not in ['#E3F2FD','#E8F5E9','#FFF3E0'] else '#333')
            if is_diff:
                ax.text(9.7, y + 0.65, '*', fontsize=14, color='#E74C3C', fontweight='bold')
            if i < len(components) - 1:
                ax.annotate('', xy=(5, y), xytext=(5, y - 0.25),
                            arrowprops=dict(arrowstyle='<-', color='#666', lw=1.5))

    vllm_components = [
        ('API Server (OpenAI-compatible)', '#4A90D9'),
        ('LLMEngine / AsyncLLMEngine', '#4A90D9'),
        ('Scheduler (FCFS + Preemption)', '#2980B9'),
        ('BlockManager (PagedAttention)', '#2471A3'),
        ('GPU/Ray Executor', '#1F618D'),
        ('Model Runner (CUDAGraph)', '#1A5276'),
        ('Attention Backend (FlashAttn/FlashInfer)', '#154360'),
        ('Sampler + LogitsProcessor', '#4A90D9'),
        ('GPU KV Cache (Paged Blocks)', '#1B4F72'),
    ]
    # Indices that differ significantly: Scheduler(2), BlockManager(3), Executor(4), Attention(6), Sampler(7)
    vllm_diffs = {2, 3, 4, 6, 7}

    sglang_components = [
        ('SGLang Frontend DSL + HTTP Router', '#27AE60'),
        ('TokenizerManager + DetokenizerManager', '#27AE60'),
        ('Scheduler (Cache-Aware Ordering)', '#229954'),
        ('RadixCache (Radix Tree KV Mgmt)', '#1E8449'),
        ('TpModelWorker (Custom ZMQ)', '#196F3D'),
        ('ModelRunner (CUDAGraph + torch.compile)', '#186A3B'),
        ('FlashInfer Backend (Tight Integration)', '#145A32'),
        ('Sampling + Jump-Forward Constrained Decoding', '#27AE60'),
        ('GPU KV Cache (Radix Tree Managed)', '#0E6655'),
    ]
    sglang_diffs = {0, 2, 3, 4, 6, 7}

    draw_stack(ax1, 'vLLM Architecture', '#4A90D9', vllm_components, vllm_diffs)
    draw_stack(ax2, 'SGLang Architecture', '#27AE60', sglang_components, sglang_diffs)

    # Legend
    fig.text(0.5, 0.02, '* Red border = Key architectural difference between the two systems',
             ha='center', fontsize=12, color='#E74C3C', fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFEBEE', edgecolor='#E74C3C'))

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.savefig('architecture_side_by_side_highlighted.png', dpi=150, bbox_inches='tight')
    plt.show()

draw_side_by_side_architecture_highlighted()

In [ ]:
# Demo 1: Side-by-side Architecture Diagrams

def draw_architecture_comparison():
    """Draw side-by-side architecture diagrams for vLLM and SGLang."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 12))
    
    # --- vLLM Architecture ---
    ax1.set_xlim(0, 10)
    ax1.set_ylim(0, 14)
    ax1.set_title('vLLM Architecture', fontsize=16, fontweight='bold', color='#2196F3')
    ax1.axis('off')
    
    vllm_components = [
        # (x, y, width, height, label, color)
        (1, 12, 8, 1.2, 'API Server\n(OpenAI-compatible)', '#E3F2FD'),
        (1, 10, 8, 1.2, 'LLMEngine / AsyncLLMEngine', '#BBDEFB'),
        (1, 8, 3.5, 1.2, 'Scheduler\n(SequenceGroup)', '#90CAF9'),
        (5.5, 8, 3.5, 1.2, 'Block Manager\n(PagedAttention)', '#90CAF9'),
        (1, 6, 8, 1.2, 'Model Executor\n(Ray/MP Workers)', '#64B5F6'),
        (1, 4, 3.5, 1.2, 'Model Runner\n(torch forward)', '#42A5F5'),
        (5.5, 4, 3.5, 1.2, 'Attention Backend\n(FlashAttention/etc)', '#42A5F5'),
        (1, 2, 8, 1.2, 'Sampler / Logits Processor', '#2196F3'),
        (1, 0.3, 8, 1.0, 'GPU KV Cache (Paged Blocks)', '#1565C0'),
    ]
    
    for x, y, w, h, label, color in vllm_components:
        rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                              facecolor=color, edgecolor='#0D47A1', linewidth=1.5)
        ax1.add_patch(rect)
        ax1.text(x + w/2, y + h/2, label, ha='center', va='center',
                fontsize=9, fontweight='bold')
    
    # Arrows
    for y_start, y_end in [(12, 11.2), (10, 9.2), (8, 7.2), (6, 5.2), (4, 3.2)]:
        ax1.annotate('', xy=(5, y_end + 1), xytext=(5, y_start),
                    arrowprops=dict(arrowstyle='->', color='#0D47A1', lw=2))
    
    # --- SGLang Architecture ---
    ax2.set_xlim(0, 10)
    ax2.set_ylim(0, 14)
    ax2.set_title('SGLang Architecture', fontsize=16, fontweight='bold', color='#4CAF50')
    ax2.axis('off')
    
    sglang_components = [
        (1, 12, 8, 1.2, 'SGLang Frontend DSL\n(Python Language)', '#E8F5E9'),
        (1, 10, 8, 1.2, 'HTTP Router / TokenizerManager', '#C8E6C9'),
        (1, 8, 3.5, 1.2, 'Scheduler\n(Cache-Aware)', '#A5D6A7'),
        (5.5, 8, 3.5, 1.2, 'Radix Cache\n(RadixAttention)', '#A5D6A7'),
        (1, 6, 8, 1.2, 'TpModelWorker\n(Tensor Parallel)', '#81C784'),
        (1, 4, 3.5, 1.2, 'ModelRunner\n(torch forward)', '#66BB6A'),
        (5.5, 4, 3.5, 1.2, 'FlashInfer Backend\n(Attention Kernels)', '#66BB6A'),
        (1, 2, 8, 1.2, 'Sampling / Constrained Decoding', '#4CAF50'),
        (1, 0.3, 8, 1.0, 'GPU KV Cache (Radix Tree Managed)', '#2E7D32'),
    ]
    
    for x, y, w, h, label, color in sglang_components:
        rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                              facecolor=color, edgecolor='#1B5E20', linewidth=1.5)
        ax2.add_patch(rect)
        ax2.text(x + w/2, y + h/2, label, ha='center', va='center',
                fontsize=9, fontweight='bold')
    
    for y_start, y_end in [(12, 11.2), (10, 9.2), (8, 7.2), (6, 5.2), (4, 3.2)]:
        ax2.annotate('', xy=(5, y_end + 1), xytext=(5, y_start),
                    arrowprops=dict(arrowstyle='->', color='#1B5E20', lw=2))
    
    plt.tight_layout()
    plt.savefig('architecture_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Architecture diagrams saved to architecture_comparison.png")

draw_architecture_comparison()

## 2. Component-Level Comparison

Let's build a detailed component mapping between both systems.

In [ ]:
# Demo 2: Component Mapping Table (programmatic)

component_mapping = {
    'API Layer': {
        'vLLM': 'api_server.py (FastAPI, OpenAI-compatible)',
        'SGLang': 'launch_server.py + Router (FastAPI + custom endpoints)',
        'key_diff': 'SGLang adds native DSL endpoint beyond OpenAI format'
    },
    'Engine Core': {
        'vLLM': 'LLMEngine / AsyncLLMEngine',
        'SGLang': 'TokenizerManager + Scheduler (separated concerns)',
        'key_diff': 'vLLM bundles tokenization+scheduling; SGLang separates them'
    },
    'Scheduler': {
        'vLLM': 'Scheduler (SequenceGroup-based, FCFS + preemption)',
        'SGLang': 'Scheduler (cache-aware, prefix-locality-driven)',
        'key_diff': 'SGLang scheduler considers cache locality for ordering'
    },
    'Memory Manager': {
        'vLLM': 'BlockSpaceManager (PagedAttention, block tables)',
        'SGLang': 'RadixCache (radix tree, automatic prefix sharing)',
        'key_diff': 'RadixAttention enables tree-based prefix dedup automatically'
    },
    'Worker': {
        'vLLM': 'Worker / GPUExecutor (Ray or multiprocessing)',
        'SGLang': 'TpModelWorker (custom TP, ZMQ-based communication)',
        'key_diff': 'vLLM uses Ray for distributed; SGLang uses custom ZMQ'
    },
    'Model Runner': {
        'vLLM': 'ModelRunner (CUDAGraph, eager mode)',
        'SGLang': 'ModelRunner (CUDAGraph, torch.compile support)',
        'key_diff': 'Both support CUDAGraph; SGLang adds torch.compile path'
    },
    'Attention Backend': {
        'vLLM': 'FlashAttention, FlashInfer, XFormers (pluggable)',
        'SGLang': 'FlashInfer (primary), triton kernels',
        'key_diff': 'vLLM has more backends; SGLang is tightly integrated with FlashInfer'
    },
    'Sampler': {
        'vLLM': 'Sampler + LogitsProcessor pipeline',
        'SGLang': 'Sampling + integrated constrained decoding (jump-forward)',
        'key_diff': 'SGLang natively integrates constrained decoding with jump-ahead'
    },
}

# Print as formatted table
print(f"{'Component':<20} {'vLLM':<45} {'SGLang':<45}")
print('=' * 110)
for comp, details in component_mapping.items():
    print(f"\n{comp}")
    print(f"  vLLM:   {details['vLLM']}")
    print(f"  SGLang: {details['SGLang']}")
    print(f"  Diff:   {details['key_diff']}")
    print('-' * 110)

In [ ]:
# Demo 3: Visualize component mapping as a heatmap of similarity

components = list(component_mapping.keys())
# Similarity score: 1 = very similar, 5 = very different
similarity_scores = {
    'API Layer': 3,
    'Engine Core': 4,
    'Scheduler': 5,
    'Memory Manager': 5,
    'Worker': 4,
    'Model Runner': 2,
    'Attention Backend': 3,
    'Sampler': 4,
}

fig, ax = plt.subplots(figsize=(12, 5))
scores = [similarity_scores[c] for c in components]
colors = plt.cm.RdYlGn_r(np.array(scores) / 5.0)

bars = ax.barh(components, scores, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Architectural Divergence (1=Similar, 5=Very Different)', fontsize=12)
ax.set_title('vLLM vs SGLang: Component-Level Architectural Divergence', fontsize=14, fontweight='bold')
ax.set_xlim(0, 6)

for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{score}/5', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Code Organization Comparison

Understanding how each project is structured helps grasp the design decisions.

In [ ]:
# Demo 4: Code Organization Tree Comparison

vllm_tree = {
    'vllm/': {
        'entrypoints/': ['api_server.py', 'llm.py', 'openai/'],
        'engine/': ['llm_engine.py', 'async_llm_engine.py', 'arg_utils.py'],
        'core/': ['scheduler.py', 'block_manager.py', 'block/'],
        'executor/': ['gpu_executor.py', 'ray_gpu_executor.py'],
        'worker/': ['worker.py', 'model_runner.py', 'cache_engine.py'],
        'model_executor/': ['model_loader/', 'layers/'],
        'attention/': ['backends/', 'ops/'],
        'transformers_utils/': ['tokenizer.py', 'config.py'],
        'sampling/': ['sampler.py', 'logits_processor.py'],
        'lora/': ['lora_manager.py', 'layers.py'],
        'spec_decode/': ['spec_decode_worker.py'],
    }
}

sglang_tree = {
    'python/sglang/': {
        'srt/': {
            'managers/': ['tokenizer_manager.py', 'detokenizer_manager.py',
                          'data_parallel_controller.py', 'schedule_batch.py'],
            'model_executor/': ['model_runner.py', 'forward_batch_info.py'],
            'layers/': ['attention/', 'sampler.py', 'logits_processor.py'],
            'mem_cache/': ['radix_cache.py', 'memory_pool.py',
                           'chunk_cache.py'],
            'server.py': None,
            'scheduler.py': None,
        },
        'lang/': ['frontend.py', 'interpreter.py', 'ir.py',
                  'chat_template.py'],
        'api.py': None,
        'bench_serving.py': None,
    }
}

def print_tree(tree, indent=0):
    for key, value in tree.items():
        print(' ' * indent + key)
        if isinstance(value, dict):
            print_tree(value, indent + 4)
        elif isinstance(value, list):
            for item in value:
                print(' ' * (indent + 4) + item)

print("=== vLLM Code Organization ===")
print_tree(vllm_tree)
print("\n=== SGLang Code Organization ===")
print_tree(sglang_tree)
print("\nKey observation: vLLM has a flatter structure; SGLang nests srt/ (serving runtime) and lang/ (frontend) separately.")

In [ ]:
# Demo 5: Code Size Comparison (approximate lines of code by component)

categories = ['Scheduler', 'Memory Mgmt', 'Model Runner', 'Attention',
              'Sampling', 'API/Server', 'Worker/Executor', 'Frontend DSL']
vllm_loc = [2500, 3500, 4000, 3000, 1500, 2000, 3000, 0]
sglang_loc = [2000, 2500, 3000, 2000, 1200, 1500, 2000, 3000]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, vllm_loc, width, label='vLLM', color='#2196F3', alpha=0.8)
bars2 = ax.bar(x + width/2, sglang_loc, width, label='SGLang', color='#4CAF50', alpha=0.8)

ax.set_ylabel('Approximate Lines of Code')
ax.set_title('Code Volume by Component: vLLM vs SGLang', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=30, ha='right')
ax.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height + 50,
                    f'{height:,}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()
print("Note: SGLang has a unique Frontend DSL component (~3000 LoC) with no vLLM equivalent.")
print("vLLM invests more code in memory management (PagedAttention is more complex).")

## 4. Design Philosophy Deep Dive

In [ ]:
# Demo 6: Design Philosophy Radar Chart

categories_radar = ['Memory\nEfficiency', 'Throughput\nOptimization',
                    'Prefix\nCaching', 'Frontend\nExpressiveness',
                    'Ecosystem\nBreadth', 'Ease of\nIntegration',
                    'Multi-GPU\nScaling', 'Constrained\nDecoding']

# Scores out of 10
vllm_scores = [9, 9, 7, 5, 9, 9, 9, 6]
sglang_scores = [8, 9, 10, 10, 7, 7, 8, 9]

num_vars = len(categories_radar)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

vllm_scores_plot = vllm_scores + vllm_scores[:1]
sglang_scores_plot = sglang_scores + sglang_scores[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
ax.fill(angles, vllm_scores_plot, alpha=0.15, color='#2196F3')
ax.plot(angles, vllm_scores_plot, 'o-', linewidth=2, color='#2196F3', label='vLLM')
ax.fill(angles, sglang_scores_plot, alpha=0.15, color='#4CAF50')
ax.plot(angles, sglang_scores_plot, 's-', linewidth=2, color='#4CAF50', label='SGLang')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories_radar, fontsize=10)
ax.set_ylim(0, 10)
ax.set_title('Design Philosophy Radar: vLLM vs SGLang', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1.1), fontsize=12)
ax.set_yticks([2, 4, 6, 8, 10])

plt.tight_layout()
plt.show()
print("\nKey takeaways:")
print("- vLLM excels in ecosystem breadth and ease of integration")
print("- SGLang leads in prefix caching and frontend expressiveness")
print("- Both are strong in throughput optimization")

## 5. Request Flow Tracing

Let's trace the same request through both systems to see how they handle it differently.

In [ ]:
# Demo 7: Simulate request flow through vLLM

class VLLMRequestTracer:
    """Simulates the request flow through vLLM's architecture."""
    
    def __init__(self):
        self.trace_log = []
        self.step = 0
    
    def log(self, component: str, action: str, details: str):
        self.step += 1
        entry = {'step': self.step, 'component': component,
                 'action': action, 'details': details}
        self.trace_log.append(entry)
        print(f"  [{self.step:2d}] {component:<25s} | {action:<20s} | {details}")
    
    def trace_request(self, prompt: str, max_tokens: int = 10):
        print(f"\n{'='*80}")
        print(f"vLLM Request Trace: prompt='{prompt[:40]}...' max_tokens={max_tokens}")
        print(f"{'='*80}")
        print(f"  {'Step':<6} {'Component':<25} | {'Action':<20} | Details")
        print(f"  {'-'*74}")
        
        # 1. API Server receives request
        self.log('API Server', 'receive_request', 'HTTP POST /v1/completions')
        self.log('API Server', 'validate_params', f'model, prompt ({len(prompt)} chars), max_tokens={max_tokens}')
        
        # 2. Engine processes
        self.log('LLMEngine', 'add_request', 'Create SequenceGroup with request_id')
        self.log('Tokenizer', 'tokenize', f'Encode prompt -> {len(prompt.split())*2} tokens (approx)')
        self.log('LLMEngine', 'create_seq_group', 'SequenceGroup(seqs=[Sequence], sampling_params)')
        
        # 3. Scheduler
        self.log('Scheduler', 'add_to_waiting', 'Add SequenceGroup to waiting queue')
        self.log('Scheduler', 'schedule()', 'Check GPU blocks availability')
        self.log('BlockManager', 'allocate', 'Allocate physical blocks for KV cache')
        self.log('Scheduler', 'move_to_running', 'waiting -> running queue')
        
        # 4. Execution
        self.log('Executor', 'execute_model', 'Send SchedulerOutputs to workers')
        self.log('Worker', 'prepare_input', 'Build input tensors (token_ids, positions, block_tables)')
        self.log('ModelRunner', 'forward', 'Run model forward pass (prefill)')
        self.log('AttentionBackend', 'compute', 'FlashAttention prefill kernel')
        self.log('Sampler', 'sample', 'Apply temperature, top_p -> sample token')
        
        # 5. Decode loop (simplified)
        for i in range(min(3, max_tokens)):
            self.log('Scheduler', 'schedule()', f'Decode step {i+1}: check preemption')
            self.log('ModelRunner', 'forward', f'Decode step {i+1}: single token forward')
            self.log('Sampler', 'sample', f'Decode step {i+1}: sample next token')
        
        if max_tokens > 3:
            self.log('...', '...', f'Continue for {max_tokens-3} more decode steps')
        
        # 6. Completion
        self.log('LLMEngine', 'finish_request', 'Detokenize output, create CompletionResponse')
        self.log('BlockManager', 'free', 'Free physical blocks back to pool')
        self.log('API Server', 'return_response', 'HTTP 200 with JSON response')
        
        return self.trace_log

vllm_tracer = VLLMRequestTracer()
vllm_trace = vllm_tracer.trace_request("What is the capital of France?", max_tokens=5)

In [ ]:
# Demo 8: Simulate request flow through SGLang

class SGLangRequestTracer:
    """Simulates the request flow through SGLang's architecture."""
    
    def __init__(self):
        self.trace_log = []
        self.step = 0
    
    def log(self, component: str, action: str, details: str):
        self.step += 1
        entry = {'step': self.step, 'component': component,
                 'action': action, 'details': details}
        self.trace_log.append(entry)
        print(f"  [{self.step:2d}] {component:<25s} | {action:<20s} | {details}")
    
    def trace_request(self, prompt: str, max_tokens: int = 10):
        print(f"\n{'='*80}")
        print(f"SGLang Request Trace: prompt='{prompt[:40]}...' max_tokens={max_tokens}")
        print(f"{'='*80}")
        print(f"  {'Step':<6} {'Component':<25} | {'Action':<20} | Details")
        print(f"  {'-'*74}")
        
        # 1. Server receives request
        self.log('HTTP Server', 'receive_request', 'HTTP POST /generate')
        self.log('TokenizerManager', 'tokenize', f'Encode prompt -> {len(prompt.split())*2} tokens')
        
        # 2. Radix Cache lookup (key SGLang difference!)
        self.log('RadixCache', 'match_prefix', 'Search radix tree for cached prefix')
        self.log('RadixCache', 'prefix_result', 'Found 0 cached tokens (new prompt)')
        
        # 3. Scheduler
        self.log('Scheduler', 'add_request', 'Create Req with prefix info')
        self.log('Scheduler', 'get_next_batch', 'Cache-aware: prioritize by prefix locality')
        self.log('Scheduler', 'alloc_token_slots', 'Allocate memory from pool')
        
        # 4. Execution
        self.log('TpModelWorker', 'forward_batch', 'Prepare and send to model')
        self.log('ModelRunner', 'forward_extend', 'Prefill with extend (FlashInfer ragged)')
        self.log('FlashInfer', 'attention_kernel', 'Ragged prefill attention kernel')
        self.log('Sampler', 'sample', 'Sample with optional constrained decoding')
        
        # 5. Decode loop
        for i in range(min(3, max_tokens)):
            self.log('Scheduler', 'get_next_batch', f'Decode {i+1}: overlap with next prefill')
            self.log('ModelRunner', 'forward_decode', f'Decode {i+1}: FlashInfer decode kernel')
            self.log('Sampler', 'sample', f'Decode {i+1}: sample next token')
        
        if max_tokens > 3:
            self.log('...', '...', f'Continue for {max_tokens-3} more decode steps')
        
        # 6. Completion (with radix cache update!)
        self.log('RadixCache', 'insert', 'Insert full sequence into radix tree for reuse')
        self.log('DetokenizerManager', 'detokenize', 'Decode tokens to text')
        self.log('HTTP Server', 'return_response', 'HTTP 200 with JSON response')
        
        return self.trace_log

sglang_tracer = SGLangRequestTracer()
sglang_trace = sglang_tracer.trace_request("What is the capital of France?", max_tokens=5)

In [ ]:
# Demo 9: Visualize request flow timelines side-by-side

def plot_request_flow_timeline(vllm_trace, sglang_trace):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
    
    component_colors = {
        'API': '#E3F2FD', 'Engine': '#BBDEFB', 'Scheduler': '#90CAF9',
        'Memory': '#64B5F6', 'Executor': '#42A5F5', 'Model': '#2196F3',
        'Sampler': '#1976D2', 'Server': '#E8F5E9', 'Cache': '#A5D6A7',
        'Worker': '#66BB6A', 'Other': '#BDBDBD'
    }
    
    def categorize(component):
        if 'API' in component or 'HTTP' in component or 'Server' in component:
            return 'API'
        elif 'Engine' in component or 'Tokenizer' in component:
            return 'Engine'
        elif 'Scheduler' in component or 'schedule' in component:
            return 'Scheduler'
        elif 'Block' in component or 'Radix' in component or 'Cache' in component:
            return 'Memory'
        elif 'Executor' in component or 'Worker' in component or 'Tp' in component:
            return 'Executor'
        elif 'Model' in component or 'Attention' in component or 'Flash' in component:
            return 'Model'
        elif 'Sampler' in component:
            return 'Sampler'
        return 'Other'
    
    # Plot vLLM
    for i, entry in enumerate(vllm_trace):
        cat = categorize(entry['component'])
        color = component_colors.get(cat, '#BDBDBD')
        ax1.barh(0, 1, left=i, height=0.6, color=color, edgecolor='black', linewidth=0.5)
        if i < 15:
            ax1.text(i + 0.5, -0.5, entry['component'][:12], rotation=45,
                    ha='right', fontsize=7)
    ax1.set_title('vLLM Request Flow', fontsize=14, fontweight='bold', color='#2196F3')
    ax1.set_yticks([])
    
    # Plot SGLang
    for i, entry in enumerate(sglang_trace):
        cat = categorize(entry['component'])
        color = component_colors.get(cat, '#BDBDBD')
        ax2.barh(0, 1, left=i, height=0.6, color=color, edgecolor='black', linewidth=0.5)
        if i < 15:
            ax2.text(i + 0.5, -0.5, entry['component'][:12], rotation=45,
                    ha='right', fontsize=7)
    ax2.set_title('SGLang Request Flow', fontsize=14, fontweight='bold', color='#4CAF50')
    ax2.set_yticks([])
    ax2.set_xlabel('Processing Steps')
    
    # Legend
    legend_patches = [mpatches.Patch(color=c, label=l) for l, c in component_colors.items()
                      if l != 'Server']
    fig.legend(handles=legend_patches, loc='upper right', ncol=4, fontsize=9)
    
    plt.tight_layout()
    plt.show()

plot_request_flow_timeline(vllm_trace, sglang_trace)
print("\nKey difference: SGLang has an early RadixCache lookup step")
print("that vLLM doesn't have. This enables prefix reuse from the start.")

## 6. Initialization and Startup Flow Comparison

In [ ]:
# Demo 10: Compare initialization flows

vllm_startup = [
    ('Parse arguments', 0.01),
    ('Create EngineConfig', 0.02),
    ('Initialize tokenizer', 0.5),
    ('Initialize model (download/load)', 15.0),
    ('Initialize BlockManager', 0.1),
    ('Pre-allocate KV cache blocks', 2.0),
    ('Profile GPU memory', 1.0),
    ('Initialize CUDA graphs', 3.0),
    ('Start API server', 0.2),
    ('Ready to serve', 0.0),
]

sglang_startup = [
    ('Parse arguments', 0.01),
    ('Create ServerArgs', 0.02),
    ('Start TokenizerManager process', 0.3),
    ('Start DetokenizerManager process', 0.2),
    ('Initialize model (download/load)', 15.0),
    ('Initialize RadixCache', 0.05),
    ('Pre-allocate KV cache pool', 2.0),
    ('Profile GPU memory', 1.0),
    ('Initialize CUDA graphs', 3.0),
    ('Start scheduler process', 0.3),
    ('Start HTTP server', 0.2),
    ('Ready to serve', 0.0),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# vLLM startup
labels_v = [s[0] for s in vllm_startup]
times_v = [s[1] for s in vllm_startup]
cumulative_v = np.cumsum(times_v)
colors_v = plt.cm.Blues(np.linspace(0.3, 0.9, len(times_v)))

ax1.barh(range(len(labels_v)), times_v, color=colors_v, edgecolor='black', linewidth=0.5)
ax1.set_yticks(range(len(labels_v)))
ax1.set_yticklabels(labels_v, fontsize=9)
ax1.set_xlabel('Time (seconds)')
ax1.set_title(f'vLLM Startup ({cumulative_v[-1]:.1f}s total)', fontweight='bold', color='#2196F3')
ax1.invert_yaxis()

# SGLang startup
labels_s = [s[0] for s in sglang_startup]
times_s = [s[1] for s in sglang_startup]
cumulative_s = np.cumsum(times_s)
colors_s = plt.cm.Greens(np.linspace(0.3, 0.9, len(times_s)))

ax2.barh(range(len(labels_s)), times_s, color=colors_s, edgecolor='black', linewidth=0.5)
ax2.set_yticks(range(len(labels_s)))
ax2.set_yticklabels(labels_s, fontsize=9)
ax2.set_xlabel('Time (seconds)')
ax2.set_title(f'SGLang Startup ({cumulative_s[-1]:.1f}s total)', fontweight='bold', color='#4CAF50')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()
print("\nBoth systems spend most startup time on model loading and CUDA graph capture.")
print("SGLang uses separate processes (TokenizerManager, DetokenizerManager, Scheduler).")
print("vLLM keeps everything in the same process (or Ray actors for distributed).")

In [ ]:
# Demo 11: Multi-turn request flow comparison showing prefix caching

def trace_multi_turn():
    """Show how both systems handle a multi-turn conversation."""
    turns = [
        "System: You are a helpful assistant.",
        "User: What is Python?",
        "Assistant: Python is a programming language...",
        "User: How do I install it?",
    ]
    
    print("=" * 80)
    print("Multi-turn Conversation: Prefix Caching Impact")
    print("=" * 80)
    
    # Simulated token counts per turn
    tokens_per_turn = [12, 8, 25, 10]
    
    print("\n--- vLLM (with APC enabled) ---")
    vllm_compute = []
    for i in range(len(turns)):
        total_tokens = sum(tokens_per_turn[:i+1])
        # vLLM with automatic prefix caching: block-level prefix match
        if i == 0:
            cached = 0
        else:
            # Block-aligned caching: can only reuse full blocks (e.g., block_size=16)
            prev_tokens = sum(tokens_per_turn[:i])
            cached = (prev_tokens // 16) * 16  # Block-aligned
        compute = total_tokens - cached
        vllm_compute.append(compute)
        print(f"  Turn {i+1}: total={total_tokens} tokens, cached={cached}, compute={compute}")
    
    print("\n--- SGLang (RadixAttention) ---")
    sglang_compute = []
    for i in range(len(turns)):
        total_tokens = sum(tokens_per_turn[:i+1])
        if i == 0:
            cached = 0
        else:
            # Radix tree: token-level prefix match
            cached = sum(tokens_per_turn[:i])  # Exact prefix match
        compute = total_tokens - cached
        sglang_compute.append(compute)
        print(f"  Turn {i+1}: total={total_tokens} tokens, cached={cached}, compute={compute}")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(turns))
    width = 0.35
    ax.bar(x - width/2, vllm_compute, width, label='vLLM (block-aligned)', color='#2196F3')
    ax.bar(x + width/2, sglang_compute, width, label='SGLang (token-level)', color='#4CAF50')
    ax.set_xlabel('Conversation Turn')
    ax.set_ylabel('Tokens to Compute (prefill)')
    ax.set_title('Prefill Computation per Turn: Block vs Token-level Caching', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'Turn {i+1}' for i in range(len(turns))])
    ax.legend()
    plt.tight_layout()
    plt.show()

trace_multi_turn()

---
## Hands-on Assignment 1: Create a Component Mapping Table

Build a comprehensive component mapping between vLLM and SGLang. For each component, identify:
1. The corresponding file(s) in each codebase
2. The key class(es) involved
3. A one-sentence summary of how the implementations differ

**Expected output**: A printed table covering at least 10 components.

In [ ]:
# Assignment 1: Complete the component mapping
# Fill in the TODO fields

assignment1_mapping = [
    {
        'component': 'Request Handler',
        'vllm_file': 'vllm/entrypoints/api_server.py',
        'vllm_class': 'TODO',  # Find the main class/function
        'sglang_file': 'python/sglang/srt/server.py',
        'sglang_class': 'TODO',  # Find the main class/function
        'difference': 'TODO'  # How do they differ?
    },
    {
        'component': 'Sequence Representation',
        'vllm_file': 'vllm/sequence.py',
        'vllm_class': 'TODO',
        'sglang_file': 'python/sglang/srt/managers/schedule_batch.py',
        'sglang_class': 'TODO',
        'difference': 'TODO'
    },
    {
        'component': 'Scheduler',
        'vllm_file': 'TODO',
        'vllm_class': 'TODO',
        'sglang_file': 'TODO',
        'sglang_class': 'TODO',
        'difference': 'TODO'
    },
    {
        'component': 'KV Cache Manager',
        'vllm_file': 'TODO',
        'vllm_class': 'TODO',
        'sglang_file': 'TODO',
        'sglang_class': 'TODO',
        'difference': 'TODO'
    },
    {
        'component': 'Model Loader',
        'vllm_file': 'TODO',
        'vllm_class': 'TODO',
        'sglang_file': 'TODO',
        'sglang_class': 'TODO',
        'difference': 'TODO'
    },
    # Add at least 5 more components...
    # Hint: Think about sampling, LoRA, speculative decoding,
    # tokenization, attention, distributed communication
]

# Print your table
print(f"{'Component':<25} {'vLLM File':<40} {'SGLang File':<45}")
print('=' * 110)
for entry in assignment1_mapping:
    print(f"{entry['component']:<25} {entry['vllm_file']:<40} {entry['sglang_file']:<45}")
    print(f"  vLLM class: {entry['vllm_class']}")
    print(f"  SGLang class: {entry['sglang_class']}")
    print(f"  Difference: {entry['difference']}")
    print('-' * 110)

---
## Hands-on Assignment 2: Identify 5 Architectural Differences with Code Evidence

For each difference, provide:
1. The architectural difference
2. Code snippets (pseudocode or actual) from both systems
3. Why this design choice was made
4. The trade-off involved

In [ ]:
# Assignment 2: Document architectural differences

architectural_differences = [
    {
        'name': 'Cache Architecture',
        'vllm_approach': '''
# vLLM: Block-based caching (PagedAttention)
# From vllm/core/block_manager.py (pseudocode)
class BlockSpaceManager:
    def allocate(self, seq_group):
        num_blocks = ceil(seq_len / block_size)
        physical_blocks = self.gpu_allocator.allocate(num_blocks)
        block_table[seq_id] = physical_blocks  # Maps logical -> physical
''',
        'sglang_approach': '''
# SGLang: Radix tree caching (RadixAttention)
# From python/sglang/srt/mem_cache/radix_cache.py (pseudocode)
class RadixCache:
    def match_prefix(self, token_ids):
        node = self.root
        for token in token_ids:
            if token in node.children:
                node = node.children[token]
            else:
                break
        return node.cached_kv_indices  # Exact prefix match
''',
        'why': 'TODO - explain the design motivation',
        'tradeoff': 'TODO - what is gained/lost'
    },
    {
        'name': 'Process Architecture',
        'vllm_approach': 'TODO - describe vLLM process model',
        'sglang_approach': 'TODO - describe SGLang process model',
        'why': 'TODO',
        'tradeoff': 'TODO'
    },
    {
        'name': 'Scheduling Strategy',
        'vllm_approach': 'TODO',
        'sglang_approach': 'TODO',
        'why': 'TODO',
        'tradeoff': 'TODO'
    },
    {
        'name': 'Frontend Design',
        'vllm_approach': 'TODO',
        'sglang_approach': 'TODO',
        'why': 'TODO',
        'tradeoff': 'TODO'
    },
    {
        'name': 'Attention Kernel Integration',
        'vllm_approach': 'TODO',
        'sglang_approach': 'TODO',
        'why': 'TODO',
        'tradeoff': 'TODO'
    },
]

for i, diff in enumerate(architectural_differences, 1):
    print(f"\n{'='*60}")
    print(f"Difference {i}: {diff['name']}")
    print(f"{'='*60}")
    print(f"\nvLLM Approach:\n{diff['vllm_approach']}")
    print(f"\nSGLang Approach:\n{diff['sglang_approach']}")
    print(f"\nMotivation: {diff['why']}")
    print(f"Trade-off: {diff['tradeoff']}")

---
## Hands-on Assignment 3: Write a Comparison Essay with Diagrams

Create a visual comparison summary. Complete the code below to:
1. Generate a comparison infographic
2. Write a structured comparison analysis
3. Propose improvements each system could adopt from the other

In [ ]:
# Assignment 3: Create a comparison infographic

def create_comparison_infographic():
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('vLLM vs SGLang: Architecture Comparison Summary',
                 fontsize=16, fontweight='bold')
    
    comparison_data = [
        ('Design Philosophy', ['Systems-first', 'Co-design'], [9, 8]),
        ('Memory Management', ['PagedAttention', 'RadixAttention'], [8, 9]),
        ('Scheduling', ['FCFS+Preempt', 'Cache-aware'], [7, 9]),
        ('Ecosystem', ['Broad support', 'Growing'], [9, 7]),
        ('Frontend', ['OpenAI API', 'DSL + API'], [7, 10]),
        ('Extensibility', ['Plugin-based', 'Integrated'], [8, 7]),
    ]
    
    for idx, (title, labels, scores) in enumerate(comparison_data):
        ax = axes[idx // 3][idx % 3]
        colors = ['#2196F3', '#4CAF50']
        bars = ax.bar(['vLLM', 'SGLang'], scores, color=colors, edgecolor='black')
        ax.set_title(title, fontweight='bold')
        ax.set_ylim(0, 11)
        ax.set_ylabel('Score (1-10)')
        
        for bar, label, score in zip(bars, labels, scores):
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
                    f'{score}/10\n({label})', ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    plt.savefig('comparison_infographic.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Infographic saved to comparison_infographic.png")

create_comparison_infographic()

# TODO: Write your comparison essay below
essay = """
# Architecture Comparison: vLLM vs SGLang

## 1. Core Philosophy
TODO: Write 2-3 paragraphs comparing the design philosophies

## 2. Key Architectural Differences
TODO: Discuss the 3 most impactful architectural differences

## 3. Cross-pollination Opportunities
TODO: What could vLLM adopt from SGLang? What could SGLang adopt from vLLM?

## 4. Conclusion
TODO: Summarize which system is better for which use cases
"""
print(essay)

In [ ]:
# Assignment 3 (continued): Propose cross-pollination improvements

improvements = {
    'vLLM could adopt from SGLang': [
        'TODO: Improvement 1',
        'TODO: Improvement 2',
        'TODO: Improvement 3',
    ],
    'SGLang could adopt from vLLM': [
        'TODO: Improvement 1',
        'TODO: Improvement 2',
        'TODO: Improvement 3',
    ]
}

for direction, items in improvements.items():
    print(f"\n{direction}:")
    for i, item in enumerate(items, 1):
        print(f"  {i}. {item}")

print("\n--- End of Chapter 10.1 ---")
print("Next: Chapter 10.2 - Scheduling Strategy Comparison")

## Summary

In this chapter, we explored:

| Topic | Key Insight |
|-------|-------------|
| **Design Philosophy** | vLLM is systems-first; SGLang co-designs frontend + backend |
| **Component Mapping** | Most components have counterparts, but differ in implementation details |
| **Request Flow** | SGLang adds RadixCache lookup early; vLLM uses block-level matching |
| **Startup** | Both dominated by model loading; SGLang uses multi-process architecture |
| **Multi-turn** | SGLang's token-level caching provides finer-grained prefix reuse |

Continue to **Chapter 10.2** for a deep dive into scheduling strategy comparison.